# 02 — Exploratory Data Analysis (EDA)
### PM Accelerator — Weather Trend Forecasting Assessment

This notebook produces **6 required visualizations** saved to `outputs/figures/`:
1. Temperature over time (daily global mean — line plot)
2. Precipitation distribution (histogram + KDE)
3. Correlation heatmap (all numeric columns)
4. Country-wise average temperature (top 20 — horizontal bar)
5. Box plot by continent (temperature grouped by continent)
6. Wind speed vs temperature scatter (colored by precipitation)

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from src.utils import DATA_DIR, FIGURES_DIR, apply_plot_style, save_figure, country_to_continent

apply_plot_style()
%matplotlib inline

In [2]:
# Load data (unscaled for meaningful plots)
df = pd.read_csv(
    os.path.join(DATA_DIR, 'cleaned_weather_unscaled.csv'),
    parse_dates=['last_updated'],
)
df.set_index('last_updated', inplace=True)
df.sort_index(inplace=True)
print(f'Shape: {df.shape}')
df.head()

Shape: (130783, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,temperature_celsius,temperature_fahrenheit,condition_text,wind_mph,...,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination,is_outlier
last_updated,,,,,,,,,,,,,,,,,,,,,
2024-05-16 01:45:00,United States of America,Washington Park,46.60,-120.49,America/Los_Angeles,1715849100,16.1,61.0,Clear,4.3,...,7.1,1,1,05:26 AM,08:31 PM,01:36 PM,02:52 AM,Waxing Gibbous,55,False
2024-05-16 02:45:00,El Salvador,San Salvador,13.71,-89.20,America/El_Salvador,1715849100,26.0,78.8,Moderate or heavy rain with thunder,2.2,...,28.1,2,2,05:30 AM,06:16 PM,01:00 PM,01:02 AM,Waxing Gibbous,55,True
2024-05-16 02:45:00,Costa Rica,San Juan,9.97,-84.08,America/Costa_Rica,1715849100,21.0,69.8,Fog,2.2,...,23.3,2,2,05:15 AM,05:51 PM,12:42 PM,12:37 AM,Waxing Gibbous,55,True
2024-05-16 02:45:00,Guatemala,Guatemala City,14.62,-90.53,America/Guatemala,1715849100,20.0,68.0,Mist,13.6,...,178.1,4,10,05:34 AM,06:23 PM,01:05 PM,01:09 AM,Waxing Gibbous,55,True
2024-05-16 02:45:00,Nicaragua,Managua,12.15,-86.27,America/Managua,1715849100,27.2,80.9,Patchy rain nearby,3.6,...,14.7,1,1,05:21 AM,06:02 PM,12:49 PM,12:49 AM,Waxing Gibbous,55,False


## Plot 1 — Temperature Over Time
Line plot of daily global mean temperature.

In [3]:
daily_temp = df['temperature_celsius'].resample('D').mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_temp.index, daily_temp.values, color='#E74C3C', linewidth=1.0, alpha=0.85)
ax.fill_between(daily_temp.index, daily_temp.values, alpha=0.15, color='#E74C3C')
ax.set_title('Daily Global Mean Temperature Over Time', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.grid(True, alpha=0.3)
save_figure(fig, '01_temperature_over_time.png')
plt.show()

Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\01_temperature_over_time.png


## Plot 2 — Precipitation Distribution
Histogram with KDE overlay (capped at 99th percentile for readability).

In [4]:
fig, ax = plt.subplots(figsize=(12, 5))
precip_data = df['precip_mm'].dropna()
cap = precip_data.quantile(0.99)
precip_capped = precip_data[precip_data <= cap]

ax.hist(precip_capped, bins=80, color='#3498DB', edgecolor='white', alpha=0.7, density=True, label='Histogram')
precip_capped.plot.kde(ax=ax, color='#E74C3C', linewidth=2, label='KDE')
ax.set_title('Precipitation Distribution (capped at 99th percentile)', fontsize=16, fontweight='bold')
ax.set_xlabel('Precipitation (mm)')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)
save_figure(fig, '02_precipitation_distribution.png')
plt.show()

Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\02_precipitation_distribution.png


## Plot 3 — Correlation Heatmap
Annotated heatmap of all numeric feature correlations.

In [5]:
numeric_df = df.select_dtypes(include=[np.number])
keep = numeric_df.columns[numeric_df.isnull().mean() < 0.3]
corr = numeric_df[keep].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.1f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            annot_kws={'size': 7}, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
save_figure(fig, '03_correlation_heatmap.png')
plt.show()

Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\03_correlation_heatmap.png


## Plot 4 — Country-Wise Average Temperature (Top 20)
Horizontal bar chart of the hottest 20 countries by mean temperature.

In [6]:
country_temp = (
    df.groupby('country')['temperature_celsius']
    .mean()
    .sort_values(ascending=True)
    .tail(20)
)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.95, len(country_temp)))
ax.barh(country_temp.index, country_temp.values, color=colors, edgecolor='white')
ax.set_title('Top 20 Countries by Average Temperature', fontsize=16, fontweight='bold')
ax.set_xlabel('Average Temperature (°C)')
ax.set_ylabel('Country')
for i, (val, name) in enumerate(zip(country_temp.values, country_temp.index)):
    ax.text(val + 0.3, i, f'{val:.1f}°C', va='center', fontsize=9)
ax.grid(True, axis='x', alpha=0.3)
save_figure(fig, '04_country_avg_temperature.png')
plt.show()

Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\04_country_avg_temperature.png


## Plot 5 — Temperature Box Plot by Continent
Temperature distribution grouped by continent, derived from country names.

In [7]:
print('Mapping countries to continents...')
unique_countries = df['country'].unique()
continent_cache = {c: country_to_continent(c) for c in unique_countries}
df['continent'] = df['country'].map(continent_cache)

df_cont = df[df['continent'] != 'Unknown'].copy()

fig, ax = plt.subplots(figsize=(12, 6))
continent_order = (
    df_cont.groupby('continent')['temperature_celsius']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)
palette = sns.color_palette('Set2', len(continent_order))
sns.boxplot(
    data=df_cont, x='continent', y='temperature_celsius',
    order=continent_order, palette=palette, ax=ax,
    fliersize=1, linewidth=1.2,
)
ax.set_title('Temperature Distribution by Continent', fontsize=16, fontweight='bold')
ax.set_xlabel('Continent')
ax.set_ylabel('Temperature (°C)')
ax.grid(True, axis='y', alpha=0.3)
save_figure(fig, '05_temperature_by_continent.png')
plt.show()

Mapping countries to continents...
Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\05_temperature_by_continent.png


## Plot 6 — Wind Speed vs Temperature Scatter
Scatter plot colored by precipitation (mm) using a colormap.

In [8]:
sample = df.sample(n=min(15000, len(df)), random_state=42)

fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(
    sample['wind_mph'],
    sample['temperature_celsius'],
    c=sample['precip_mm'],
    cmap='YlGnBu',
    alpha=0.5, s=8, edgecolors='none',
    vmin=0, vmax=sample['precip_mm'].quantile(0.95),
)
cbar = plt.colorbar(scatter, ax=ax, shrink=0.8)
cbar.set_label('Precipitation (mm)')
ax.set_title('Wind Speed vs Temperature (colored by Precipitation)', fontsize=16, fontweight='bold')
ax.set_xlabel('Wind Speed (mph)')
ax.set_ylabel('Temperature (°C)')
ax.grid(True, alpha=0.3)
save_figure(fig, '06_wind_vs_temperature_scatter.png')
plt.show()

print('\n✅ EDA complete! 6 plots saved to outputs/figures/')

Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\06_wind_vs_temperature_scatter.png

✅ EDA complete! 6 plots saved to outputs/figures/
